# 04. String DP (Two-String and Palindrome DP)

State is usually `dp[i][j]` = answer for the **first `i` chars of A and first `j` chars of B** (or the substring `i..j` of one string).

**Learning order:** LCS → EDIT DISTANCE → PALINDROMIC SUBSEQUENCE → PALINDROMIC SUBSTRINGS → DISTINCT SUBSEQUENCES → INTERLEAVING → WILDCARD → INTERVIEW THINKING

## 1. Longest Common Subsequence (LCS)

`dp[i][j]` = LCS length of `a[:i]` and `b[:j]`.
- If `a[i-1] == b[j-1]`: they match, `dp[i][j] = dp[i-1][j-1] + 1`.
- Else: skip one char from either string: `max(dp[i-1][j], dp[i][j-1])`.

Real life: `diff` tools, DNA comparison, version control. Time/space O(m·n); space can be reduced to O(min(m, n)) with two rows.

In [1]:
def lcs(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[m][n]

print(lcs("abcde", "ace"), lcs("abc", "def"))   # 3 0

3 0


## 2. Edit Distance (Levenshtein)

Minimum operations (insert, delete, replace) to turn `a` into `b`. `dp[i][j]` = distance between `a[:i]` and `b[:j]`.
- Chars equal: `dp[i-1][j-1]` (no cost).
- Else: `1 + min(dp[i-1][j-1]` (replace), `dp[i-1][j]` (delete), `dp[i][j-1]` (insert)`)`.
- Base: `dp[i][0] = i`, `dp[0][j] = j`.

Real life: spell checkers, search "did you mean...".

In [2]:
def edit_distance(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j - 1], dp[i - 1][j], dp[i][j - 1])
    return dp[m][n]

print(edit_distance("horse", "ros"), edit_distance("intention", "execution"))   # 3 5

3 5


## 3. Longest Palindromic Subsequence

A palindrome reads the same reversed, so the LPS of `s` = **LCS of `s` and `reversed(s)`**.

In [3]:
def longest_palindrome_subseq(s):
    return lcs(s, s[::-1])

print(longest_palindrome_subseq("bbbab"), longest_palindrome_subseq("cbbd"))   # 4 2

4 2


## 4. Palindromic Substrings (count) and Longest Palindromic Substring

Substrings must be contiguous. **Expand around every center** (2n - 1 centers: each character, and each gap between characters). O(n²) time, O(1) space. (A DP table `is_pal[i][j]` also works but uses O(n²) memory. Manacher's algorithm is O(n).)

In [4]:
def count_substrings(s):
    count = 0
    for center in range(2 * len(s) - 1):
        l, r = center // 2, center // 2 + center % 2
        while l >= 0 and r < len(s) and s[l] == s[r]:
            count += 1
            l -= 1
            r += 1
    return count

def longest_palindrome(s):
    best = ""
    for center in range(2 * len(s) - 1):
        l, r = center // 2, center // 2 + center % 2
        while l >= 0 and r < len(s) and s[l] == s[r]:
            l -= 1
            r += 1
        if r - l - 1 > len(best):
            best = s[l + 1:r]
    return best

print(count_substrings("abc"), count_substrings("aaa"), len(longest_palindrome("babad")))   # 3 6 3

3 6 3


## 5. Distinct Subsequences

Number of ways `s` contains `t` as a **subsequence**. `dp[j]` = ways to form `t[:j]` from the part of `s` seen so far. For each char of `s`, going `j` **downwards**: if it equals `t[j-1]`, `dp[j] += dp[j-1]`. Base `dp[0] = 1` (empty target).

In [5]:
def num_distinct(s, t):
    dp = [1] + [0] * len(t)
    for ch in s:
        for j in range(len(t), 0, -1):
            if ch == t[j - 1]:
                dp[j] += dp[j - 1]
    return dp[len(t)]

print(num_distinct("rabbbit", "rabbit"), num_distinct("babgbag", "bag"))   # 3 5

3 5


## 6. Interleaving String

Is `s3` formed by interleaving `s1` and `s2` (keeping each string's order)? `dp[i][j]` = can the first `i + j` characters of `s3` be formed from `s1[:i]` and `s2[:j]`. The last character comes from `s1` or from `s2`.

In [6]:
def is_interleave(s1, s2, s3):
    m, n = len(s1), len(s2)
    if m + n != len(s3):
        return False
    dp = [[False] * (n + 1) for _ in range(m + 1)]
    dp[0][0] = True
    for i in range(m + 1):
        for j in range(n + 1):
            if i > 0 and dp[i - 1][j] and s1[i - 1] == s3[i + j - 1]:
                dp[i][j] = True
            if j > 0 and dp[i][j - 1] and s2[j - 1] == s3[i + j - 1]:
                dp[i][j] = True
    return dp[m][n]

print(is_interleave("aabcc", "dbbca", "aadbbcbcac"), is_interleave("aabcc", "dbbca", "aadbbbaccc"))   # True False

True False


## 7. Wildcard Matching

`?` matches any one character, `*` matches any sequence (including empty). `dp[i][j]` = does `s[:i]` match `p[:j]`.
- `p[j-1]` is a letter or `?`: needs `dp[i-1][j-1]` and a char match.
- `p[j-1] == '*'`: matches empty (`dp[i][j-1]`) **or** one more char (`dp[i-1][j]`).

In [7]:
def is_match(s, p):
    m, n = len(s), len(p)
    dp = [[False] * (n + 1) for _ in range(m + 1)]
    dp[0][0] = True
    for j in range(1, n + 1):
        if p[j - 1] == "*":
            dp[0][j] = dp[0][j - 1]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if p[j - 1] == "*":
                dp[i][j] = dp[i][j - 1] or dp[i - 1][j]
            elif p[j - 1] == "?" or p[j - 1] == s[i - 1]:
                dp[i][j] = dp[i - 1][j - 1]
    return dp[m][n]

print(is_match("aa", "*"), is_match("cb", "?a"), is_match("adceb", "*a*b"))   # True False True

True False True


## 8. Interview thinking

**Signals:** two strings and "longest common", "minimum edits", "match with a pattern", "is it a subsequence/interleaving", palindromes with deletions.

**Say:** "Let `dp[i][j]` be the answer for prefixes `a[:i]` and `b[:j]`. If the last characters match ..., otherwise I take the best of ... O(m·n) time, and I can keep two rows for O(n) space."

**Common mistakes:** off-by-one (`a[i-1]` vs `a[i]`), missing the base row/column, mixing subsequence (not contiguous) with substring (contiguous), wrong direction in 1D optimizations, trying to recover the **string** (needs the full table to backtrack).

## Quick Revision

- LCS: match → diag + 1, else max(up, left).
- Edit distance: `1 + min(replace, delete, insert)`, base `dp[i][0] = i`.
- LPS = LCS with the reverse. Palindromic substring: expand around centers.
- Distinct subsequences: 1D, `j` downwards.
- Wildcard: `*` = empty or one more char.